# Nemotron-3-Nano-30B QLoRA training (cloud / off-cluster)

Portable version of `hpc/nem_chained.slurm` for training **outside the KU HPC cluster**.
Trains a rank-32 LoRA adapter on `data/train_deterministic_v7.jsonl` (6771 records).

## Hardware requirements
The base model is 30B params (MoE, 3.5B active). With 4-bit QLoRA the weights take ~17-20 GB plus activations/optimizer, so you need **>= 40 GB VRAM**:

| Platform | GPU | Works? |
|---|---|---|
| Kaggle | P100 16GB / 2x T4 (16GB each) | NO — too small, no bf16, model won't fit even 4-bit + training overhead |
| Colab Pro/Pro+ | A100 40GB | YES (tight; seq 2048, batch 1, grad accum 8) |
| RunPod / Vast.ai / Lambda | L40S 48GB, A100 80GB, H100 80GB | YES (recommended; ~$0.8-2.5/h, full 500-step run in 4-8h) |
| KU HPC | RTX PRO 6000 48GB | YES (reference setup) |

Checkpoints save every 50 steps to `OUT_PATH`; the run resumes automatically from the latest checkpoint, so 12h session limits are fine — just rerun the notebook with persistent storage (e.g. Colab Drive mount or RunPod volume).

In [ ]:
# 1. Dependencies
%pip install -q unsloth unsloth_zoo  # with deps: brings peft, trl, xformers (which sets the torch version), etc.
# xformers upgrades torch; torchvision must be brought along to the matching version
%pip install -q --upgrade torchvision
%pip install -q "transformers==4.57.6" "tokenizers==0.22.2" "datasets==4.3.0" bitsandbytes einops
# Nemotron's Mamba-2 layers: these compile against the installed torch, so build
# isolation must be off (otherwise pip builds in a clean env without torch and fails).
%pip install -q ninja packaging
%pip install -q causal-conv1d --no-build-isolation
%pip install -q mamba-ssm --no-build-isolation
# If the mamba-ssm build still fails, use a prebuilt wheel matching your torch/python version, e.g.:
# %pip install https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl
print('Deps installed — now RESTART THE KERNEL before running the next cell.')

In [ ]:
# 2. Config
import unsloth  # must be imported before trl/transformers for its patches
import os

MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"  # downloads ~60GB from HF
REPO_URL = "https://github.com/alrobles/nemotron-eco-reasoner.git"
DATA_PATH = "nemotron-eco-reasoner/data/train_deterministic_v7.jsonl"
OUT_PATH = "outputs/deterministic_v7"  # put this on persistent storage (Drive / volume) to survive session limits

SEQ_LEN = 2048
RANK = 32          # competition max
GRAD_ACCUM = 8
TARGET_TOTAL = 500

os.environ['TORCH_COMPILE_DISABLE'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.makedirs(OUT_PATH, exist_ok=True)

if not os.path.exists("nemotron-eco-reasoner"):
    !git clone --depth 1 {REPO_URL}

import torch
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu_name} ({vram_gb:.1f}GB) bf16={bf16_ok}")
assert vram_gb >= 38, f"Need >=40GB VRAM for 30B QLoRA; got {vram_gb:.1f}GB. See hardware table above."

In [ ]:
# 3. Load model 4-bit (QLoRA) and patch the MoE forward
# The stock NemotronH MoE dispatch is incompatible with 4-bit + grad checkpointing;
# this dense per-expert dispatch (from hpc/nem_chained.slurm) fixes it.
from unsloth import FastLanguageModel

model, tok = FastLanguageModel.from_pretrained(
    MODEL_ID, max_seq_length=SEQ_LEN, load_in_4bit=True, trust_remote_code=True)

import types as _types
patched = 0
for module in model.modules():
    if not hasattr(module, 'moe') or not hasattr(module, 'experts'): continue
    if not callable(module.moe): continue
    def mp(mod):
        def pm(_self, h, ti, tw):
            h = h.view(-1, h.size(-1)); ft = ti.view(-1); h = h.repeat_interleave(ti.shape[-1], dim=0)
            fh = torch.zeros_like(h); dt = fh.dtype
            for i, el in enumerate(mod.experts):
                m = (ft == i).nonzero(as_tuple=True)[0]
                if m.numel() == 0: continue
                eh = h[m]; eo = el(eh); w = tw.view(-1)[m]; wo = eo * w.unsqueeze(-1)
                if wo.dtype != dt: wo = wo.to(dt)
                fh.index_add_(0, m, wo)
            tk = ti.shape[-1]; fh = fh.view(-1, tk, fh.size(-1)).sum(dim=1)
            return fh
        return pm
    module.moe = _types.MethodType(mp(module), module); patched += 1
print(f"MoE patched: {patched} layers")

model = FastLanguageModel.get_peft_model(
    model, r=RANK, lora_alpha=RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth')
model.print_trainable_parameters()

In [ ]:
# 4. Dataset + resume from latest checkpoint (if any)
import glob, json
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": DATA_PATH}, split="train")
def fmt(ex):
    ms = ex.get("messages")
    if not ms:
        ms = [{"role": "user", "content": ex.get("prompt", "?")},
              {"role": "assistant", "content": ex.get("answer", "?")}]
    return {"text": tok.apply_chat_template(ms, tokenize=False, add_generation_prompt=False)}
ds = ds.map(fmt)
print(f"{len(ds)} training records")

ckpts = sorted(glob.glob(os.path.join(OUT_PATH, "checkpoint-*", "trainer_state.json")), key=os.path.getmtime)
RESUME_CKPT = os.path.dirname(ckpts[-1]) if ckpts else None
global_step = json.load(open(ckpts[-1])).get("global_step", 0) if ckpts else 0
print(f"Resume: {RESUME_CKPT or 'FRESH START'} (step {global_step})")

In [ ]:
# 5. Train (same hyperparameters as the cluster run)
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir=OUT_PATH, max_steps=TARGET_TOTAL,
        per_device_train_batch_size=1, gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=1e-4,
        # trl>=0.20 renamed max_seq_length -> max_length
        **({'max_length': SEQ_LEN} if 'max_length' in SFTConfig.__dataclass_fields__ else {'max_seq_length': SEQ_LEN}),
        warmup_steps=min(30, TARGET_TOTAL // 3), lr_scheduler_type="constant_with_warmup",
        logging_steps=10, save_steps=50, save_total_limit=10,
        bf16=bf16_ok, fp16=not bf16_ok,
        remove_unused_columns=False, report_to="none",
        dataloader_num_workers=0, packing=True),
    train_dataset=ds, processing_class=tok)

trainer.train(resume_from_checkpoint=RESUME_CKPT)
trainer.save_model(os.path.join(OUT_PATH, "final"))
tok.save_pretrained(os.path.join(OUT_PATH, "final"))

In [ ]:
# 6. Package adapter for Kaggle submission (rank <= 32 LoRA only)
import shutil
ADAPTER = os.path.join(OUT_PATH, "final")
!ls -la {ADAPTER}
shutil.make_archive("submission", "zip", ADAPTER)
print("submission.zip ready — upload to the competition (or use scripts/submit_kaggle.py)")